<a href="https://colab.research.google.com/github/jnguizo/Flowers102-MobileNet/blob/main/mobilenet_flowers102.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install torch torchvision timm pandas

In [ ]:
import time
import copy
import timm
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [ ]:
NUM_CLASSES = 102
BATCH_SIZE = 32
EPOCHS = 10

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

In [ ]:
#Chargement du dataset
train_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

test_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

train_ds = datasets.Flowers102(
    root="./data",
    split="train",
    download=True,
    transform=train_tf
)

val_ds = datasets.Flowers102(
    root="./data",
    split="val",
    download=True,
    transform=test_tf
)

test_ds = datasets.Flowers102(
    root="./data",
    split="test",
    download=True,
    transform=test_tf
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False
)

100%|██████████| 345M/345M [00:20<00:00, 17.1MB/s]
100%|██████████| 502/502 [00:00<00:00, 915kB/s]
100%|██████████| 15.0k/15.0k [00:00<00:00, 9.79MB/s]


In [ ]:
# creation du v1

def mobilenet_v1(feature_extraction=True):

    model = timm.create_model(
        "mobilenetv1_100",
        pretrained=True
    )

    if feature_extraction:
        for p in model.parameters():
            p.requires_grad = False

    in_features = model.classifier.in_features

    model.classifier = nn.Linear(
        in_features,
        NUM_CLASSES
    )

    return model

In [ ]:
#creation v2
def mobilenet_v2(feature_extraction=True):

    model = models.mobilenet_v2(
        weights=models.MobileNet_V2_Weights.DEFAULT
    )

    if feature_extraction:
        for p in model.parameters():
            p.requires_grad = False

    in_features = model.classifier[1].in_features

    model.classifier[1] = nn.Linear(
        in_features,
        NUM_CLASSES
    )

    return model

In [ ]:
#creation v3
def mobilenet_v3(feature_extraction=True):

    model = models.mobilenet_v3_large(
        weights=models.MobileNet_V3_Large_Weights.DEFAULT
    )

    if feature_extraction:
        for p in model.parameters():
            p.requires_grad = False

    in_features = model.classifier[3].in_features

    model.classifier[3] = nn.Linear(
        in_features,
        NUM_CLASSES
    )

    return model

In [ ]:
def count_params(model):
    return sum(
        p.numel()
        for p in model.parameters()
    )

def evaluate(model, loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for x, y in loader:

            x = x.to(DEVICE)
            y = y.to(DEVICE)

            pred = model(x).argmax(1)

            correct += (pred == y).sum().item()
            total += y.size(0)

    return 100 * correct / total

In [ ]:
#entrainement
def train_model(model):

    model.to(DEVICE)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        filter(
            lambda p: p.requires_grad,
            model.parameters()
        ),
        lr=1e-3
    )

    best_acc = 0

    best_weights = copy.deepcopy(
        model.state_dict()
    )

    for epoch in range(EPOCHS):

        model.train()

        for x, y in train_loader:

            x = x.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad()

            output = model(x)

            loss = criterion(output, y)

            loss.backward()

            optimizer.step()

        val_acc = evaluate(
            model,
            val_loader
        )

        if val_acc > best_acc:
            best_acc = val_acc
            best_weights = copy.deepcopy(
                model.state_dict()
            )

        print(
            f"Epoch {epoch+1} "
            f"Val Acc={val_acc:.2f}%"
        )

    model.load_state_dict(best_weights)

    return model

In [ ]:
#Temps d'inference cpu
def cpu_latency(model):

    model.cpu()
    model.eval()

    x = torch.randn(
        1,
        3,
        224,
        224
    )

    for _ in range(10):
        _ = model(x)

    start = time.perf_counter()

    with torch.no_grad():

        for _ in range(100):

            _ = model(x)

    elapsed = time.perf_counter() - start

    return elapsed * 1000 / 100


In [ ]:
results = []

models_to_test = [
    ("MobileNetV1", mobilenet_v1),
    ("MobileNetV2", mobilenet_v2),
    ("MobileNetV3", mobilenet_v3)
]

for name, builder in models_to_test:

    for feature_extraction in [True, False]:

        strategy = (
            "Feature Extraction"
            if feature_extraction
            else "Fine-Tuning"
        )

        print(
            f"\n{name} - {strategy}"
        )

        model = builder(
            feature_extraction
        )

        model = train_model(model)

        top1 = evaluate(
            model,
            test_loader
        )

        params = (
            count_params(model)
            / 1e6
        )

        latency = cpu_latency(model)

        print(f"Model      : {name}")
        print(f"Strategy   : {strategy}")
        print(f"Top1 (%)   : {top1:.2f}")
        print(f"Params (M) : {params:.2f}")
        print(f"CPU ms/img : {latency:.2f}")

        results.append([
            name,
            strategy,
            round(top1,2),
            round(params,2),
            round(latency,2)
])


MobileNetV1 - Feature Extraction
Epoch 1 Val Acc=24.61%
Epoch 2 Val Acc=48.53%
Epoch 3 Val Acc=60.29%
Epoch 4 Val Acc=63.14%
Epoch 5 Val Acc=64.41%
Epoch 6 Val Acc=69.31%
Epoch 7 Val Acc=70.59%
Epoch 8 Val Acc=72.35%
Epoch 9 Val Acc=71.18%
Epoch 10 Val Acc=73.43%
Model      : MobileNetV1
Strategy   : Feature Extraction
Top1 (%)   : 70.52
Params (M) : 3.31
CPU ms/img : 29.23

MobileNetV1 - Fine-Tuning
Epoch 1 Val Acc=4.41%
Epoch 2 Val Acc=3.24%
Epoch 3 Val Acc=7.06%
Epoch 4 Val Acc=15.29%
Epoch 5 Val Acc=21.08%
Epoch 6 Val Acc=27.94%
Epoch 7 Val Acc=29.90%
Epoch 8 Val Acc=33.63%
Epoch 9 Val Acc=43.73%
Epoch 10 Val Acc=41.18%
Model      : MobileNetV1
Strategy   : Fine-Tuning
Top1 (%)   : 40.58
Params (M) : 3.31
CPU ms/img : 23.93

MobileNetV2 - Feature Extraction
Epoch 1 Val Acc=39.71%
Epoch 2 Val Acc=59.90%
Epoch 3 Val Acc=65.88%
Epoch 4 Val Acc=69.51%
Epoch 5 Val Acc=72.06%
Epoch 6 Val Acc=72.75%
Epoch 7 Val Acc=74.90%
Epoch 8 Val Acc=75.49%
Epoch 9 Val Acc=76.37%
Epoch 10 Val Acc=77.